<a href="https://colab.research.google.com/github/w3aarush/DR_Classification_NIT_MCA_Project/blob/main/EfficientNet_KNN_Multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
# from tensorflow.keras.layers.experimental import preprocessing
from tensorflow import keras
from tensorflow.keras import layers,Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Attention
# from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, accuracy_score
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2S, preprocess_input
from google.colab.patches import cv2_imshow
import pandas as pd
import numpy as np
import seaborn as sns
# import imutils
import time
import cv2
# from cuml import SVC # for python 3.11
# from sklearn.svm import SVC

In [ ]:
# Install Kaggle API
!pip install -q kaggle

# Upload kaggle.json
from google.colab import files
files.upload()

# Setup Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Download DATASET (not competition)
!kaggle datasets download -d mariaherrerot/aptos2019

# Unzip
!unzip -q aptos2019.zip -d aptos2019

# Check files
!ls aptos2019

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mariaherrerot/aptos2019
License(s): unknown
100% 8.01G/8.01G [01:11<00:00, 121MB/s]

test.csv  test_images  train_1.csv  train_images  valid.csv  val_images


In [ ]:
base_dir = '/content/aptos2019'
train_dir = '/content/aptos2019/train_images/train_images/'
validation_dir = '/content/aptos2019/val_images/val_images/'
test_dir = '/content/aptos2019/test_images/test_images/'

In [ ]:
import os

In [ ]:
print(os.listdir(train_dir))
print(os.listdir(validation_dir))
print(os.listdir(test_dir))

['3cab32dd6ef9.png', '43f22d1be8dd.png', '89fc080f7e83.png', 'ce6f33a81ad5.png', '1b3647865779.png', '54dc6e8107cd.png', 'a01024054596.png', '8329e80c10ac.png', '66bfec8d6bcd.png', '9985375d709f.png', 'd28bd830c171.png', '542964865b1e.png', '959dc602febc.png', '61c2fbd16e38.png', 'd88c4843aec3.png', '929cd3867815.png', '20d5fdd450ae.png', '1da4a17c18c9.png', '578109578b46.png', 'a3957df90a78.png', '7c629b491d1a.png', '6c250a30593b.png', '236f56771ec6.png', '973b0facfa9b.png', 'c9485c38fdd5.png', '356304d15a5c.png', '2cdcc910778d.png', 'df6d13d04da1.png', '26cd40b57ad1.png', 'dad71ba27a9b.png', 'e33766353db2.png', 'a8c9fcdbc0be.png', '916915f01e17.png', 'da8900ac7f29.png', '6324d77cf926.png', '2b3a4a81d748.png', '5cde55f745af.png', '2ecbc2e3f239.png', 'e0b5a982a018.png', 'cbc23af521f3.png', '7ad0c4975890.png', '895fe2bfc5b6.png', 'd93b61dc8f64.png', 'e4ae1ee6aada.png', '8344c783da65.png', '1dd9adcbfff4.png', '830e297965a1.png', '98f7136d2e7a.png', 'd803598dabda.png', '7a46cfa69bae.png',

In [ ]:
NUM_CLASSES = 5
epochs = 20

In [ ]:
BATCH_SIZE = 32

In [ ]:
# img_augmentation = Sequential(
#     [
#         tf.keras.layers.RandomRotation(factor=(-0.15, 0.15)),
#         tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
#         tf.keras.layers.RandomFlip(),
#         tf.keras.layers.RandomContrast(factor=0.1),
#     ],
#     name="img_augmentation",
# )

This code defines an image augmentation pipeline using Keras's Sequential model. It applies a series of random transformations to input images:

RandomRotation(factor=(-0.15, 0.15)): Randomly rotates images by an angle within the range of -15% to +15% of 2π radians.
RandomTranslation(height_factor=0.1, width_factor=0.1): Randomly shifts images horizontally and vertically by up to 10% of their width and height, respectively.
RandomFlip(): Randomly flips images horizontally (left-right).
RandomContrast(factor=0.1): Randomly adjusts the contrast of images by a factor within the range of [1 - 0.1, 1 + 0.1] (i.e., [0.9, 1.1]).
This img_augmentation pipeline is typically used during model training to artificially increase the size and diversity of the training dataset, helping to improve the model's generalization capabilities and reduce overfitting.

In [ ]:
def extract_EfficientNetV2S_64_feature_map():
    IMG_SIZE = (224, 224)
    # Load the EfficientNetV2S base model without the top (classification) layer
    base_model = EfficientNetV2S(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')

    # Freeze the base model layers
    base_model.trainable = False

    # Create a new model on top of the base model
    inputs = keras.Input(shape=IMG_SIZE + (3,))
   # x = img_augmentation(inputs)
    x = preprocess_input(inputs) # EfficientNetV2S expects inputs in the [-1, 1] range after preprocessing
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    feature_map = layers.Dense(64, activation='relu')(x)
    # outputs = layers.Dense(5, activation='softmax')(x)
    model = keras.Model(inputs, feature_map)

    # Compile the model
    # model.compile(
    #     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    #     loss='categorical_crossentropy',
    #     metrics=['accuracy']
    # )
    return model

In [ ]:
def effnet_softmax_multiclass():
    # Use the existing feature map logic
    IMG_SIZE = (224, 224)
    base_model = EfficientNetV2S(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
    base_model.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)

    # The 'feature_map' layer we want to optimize
    feature_layer = layers.Dense(64, activation='relu', name='feature_extraction_layer')(x)

    # The 'classification_head' needed to calculate the cost function (Loss)
    # This is what allows model.fit() to actually update the weights
    outputs = layers.Dense(5, activation='softmax')(feature_layer)

    model = keras.Model(inputs, outputs)
    return model

In [ ]:
def unfreeze_model(model):
    # unfreeze the top 10 layers while leaving BatchNorm layers frozen for fine-tuning
    for layer in model.layers[-10:]:
        if not isinstance(layer, layers.BatchNormalization):
            print("executed")
            layer.trainable = True
    # optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    # model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
def test_model(model, test_batches):
    test_labels = test_batches.classes
    print("Test Lables", test_labels)
    print(test_batches.class_indices)

    # predictions = model.predict(test_batches, step=len(test_batches), verbose=0)
    predictions = model.predict(test_batches, verbose=0)


    acc = 0
    for i in range(len(test_labels)):
        actual_class = test_labels[i]
        if predictions[i][actual_class] > 0.5:
            acc += 1
    print('Accuracy:', (acc/len(test_labels))*100, "% ")
    # Convert predictions to discrete class labels for classification_report
    predicted_labels = np.argmax(predictions, axis=1)
    print('Classification Report:', classification_report(test_batches.labels, predicted_labels))

In [ ]:
def load_data():
    train = pd.read_csv('/content/aptos2019/train_1.csv', encoding='utf-8')
    test = pd.read_csv('/content/aptos2019/test.csv', encoding='utf-8')
    valid = pd.read_csv('/content/aptos2019/valid.csv')

    train_dir = '/content/aptos2019/train_images/train_images/'
    test_dir = '/content/aptos2019/test_images/test_images/'
    valid_dir = '/content/aptos2019/val_images/val_images/'

    # construct file paths directly within function:
    train['image_path'] = train_dir + train['id_code'] + '.png'
    test['image_path'] = test_dir + test['id_code'] + '.png'
    valid['image_path'] = valid_dir + valid['id_code'] + '.png'

    train['train_images'] = train['id_code'] + '.png'
    test['test_images'] = test['id_code'] + '.png'
    valid['valid_images'] = valid['id_code'] + '.png'

    train['diagnosis'] = train['diagnosis'].astype(str)
    # train['target'] = [1 if x >= 1 else 0 for x in train['diagnosis']]
    # train['target'] = train['target'].astype(str)
    test['diagnosis'] = test['diagnosis'].astype(str)
    # test['target'] = [1 if x >= 1 else 0 for x in test['diagnosis']]
    # test['target'] = test['target'].astype(str)
    valid['diagnosis'] = valid['diagnosis'].astype(str)
    # valid['target'] = [1 if x >= 1 else 0 for x in valid['diagnosis']]
    # valid['target'] = valid['target'].astype(str)

    return train, test, valid

In [ ]:
def preprocess_image(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis = 0)
    return preprocess_input(img_array)

In [ ]:
train_df, test_df, valid_df = load_data()

In [ ]:
# Initialize all data generators in the global scope
train_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
    dataframe=train_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)

valid_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
    dataframe=valid_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)

test_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
    dataframe=test_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE, shuffle=False)

Found 2930 validated image filenames belonging to 5 classes.
Found 366 validated image filenames belonging to 5 classes.
Found 366 validated image filenames belonging to 5 classes.


In [ ]:
# Instantiate the model with the classification head
train_model = effnet_softmax_multiclass()

# Compile the model
train_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-5)

# Start training
history = train_model.fit(
    train_batches,
    epochs=epochs,
    validation_data=valid_batches,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Save the model after initial training
train_model.save('effnet_softmax_multiclass_64perceptron.h5')
# train_model.save('/content/drive/MyDrive/effnet_softmax_multiclass_64perceptron.h5')
print('Model saved after initial training as initial_trained_model.h5')

After the initial training with a frozen base, we will now unfreeze the top layers of the base model for fine-tuning. This process typically involves recompiling the model with a lower learning rate and then training it for additional epochs to allow the pre-trained layers to adapt to your specific dataset.

In [ ]:
print('Unfreezing top layers for fine-tuning...')
unfreeze_model(train_model)

# Recompile the model with a much smaller learning rate for fine-tuning
train_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # A smaller learning rate is crucial for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Continuing training with unfrozen layers for fine-tuning...')
# Continue training for a few more epochs with the unfrozen layers
history_fine_tune = train_model.fit(
    train_batches,
    epochs=epochs, # You can adjust epochs if you want more fine-tuning steps
    validation_data=valid_batches,
    callbacks=[early_stopping, reduce_lr], # Continue using the same callbacks
    verbose=1
)

# Save the model after fine-tuning
train_model.save('fine_tuned_model_effnet_softmax_multiclass.h5')
# train_model.save('/content/drive/MyDrive/fine_tuned_model_effnet_softmax_multiclass.h5')
print('Model saved after fine-tuning as fine_tuned_model.h5')

# Preparing DataFrame

Now that the model has been fine-tuned, we can extract the 64-dimensional feature vectors from the `feature_extraction_layer`. We will create a new Keras model specifically for this purpose, using the `train_model`'s input and outputting the activations of the named feature layer.

In [ ]:
# Create a new model that takes the same input as train_model and outputs the 'feature_extraction_layer'
feature_extractor = keras.Model(
    inputs=train_model.inputs,
    outputs=train_model.get_layer('feature_extraction_layer').output
)

# Extract features for the training dataset
X_train_features = feature_extractor.predict(train_batches)
y_train_labels = train_batches.classes

# Extract features for the test dataset
X_test_features = feature_extractor.predict(test_batches)
y_test_labels = test_batches.classes

print('Shape of extracted training features:', X_train_features.shape)
print('Shape of extracted training labels:', y_train_labels.shape)
print('Shape of extracted test features:', X_test_features.shape)
print('Shape of extracted test labels:', y_test_labels.shape)

print('\nThese extracted features (X_train_features, X_test_features) can now be used to train and evaluate your KNN model.')

# KNN

Finally, we will use the extracted 64-dimensional features to train a K-Nearest Neighbors (KNN) classifier and evaluate its performance. This demonstrates how the learned features can be used with traditional machine learning models.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Instantiate KNN classifier with a chosen number of neighbors (e.g., 20)
knn = KNeighborsClassifier(n_neighbors=20)

# Train the KNN model using the extracted training features and labels
print('Training KNN classifier...')
knn.fit(X_train_features, y_train_labels)
print('KNN classifier trained.')

# Make predictions on the extracted test features
print('Making predictions with KNN...')
y_pred = knn.predict(X_test_features)

# Evaluate the KNN model's performance
accuracy = accuracy_score(y_test_labels, y_pred)
print(f'\nKNN Accuracy on extracted features: {accuracy:.4f}')

# Print a detailed classification report
print('\nClassification Report for KNN:')
print(classification_report(y_test_labels, y_pred))

# Previous Code

In [ ]:
if __name__ == "__main__":
    train_df, test_df, valid_df = load_data()
    model = extract_EfficientNetV2S_feature_map()
    unfreeze_model(model)

    train_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=train_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)
    valid_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=valid_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)
    test_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=test_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE, shuffle=False)

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-4)
    history = model.fit(train_batches, epochs=epochs, validation_data=valid_batches, verbose=1,callbacks=[early_stopping,reduce_lr])